# 不使用@tool的方式定义工具

## 1、举例

In [1]:
import os
from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek

load_dotenv(override=True)
from rich import print as rprint

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

llm_deepseek = ChatDeepSeek(
    model="deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    api_base=DEEPSEEK_BASE_URL,
    extra_body={"thinking":{"type":"disabled"}},
)
# 2、声明一个函数（工具）
def get_weather(city : str):
    return f"{city}天气晴朗~~"

# 3、将函数绑定在模型上
model_with_tools = llm_deepseek.bind_tools([get_weather])

# 4、调用模型
response = model_with_tools.invoke("大连的天气怎么样")
rprint(response)

AIMessage(
    content='',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 38,
            'prompt_tokens': 262,
            'total_tokens': 300,
            'completion_tokens_details': None,
            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 128},
            'prompt_cache_hit_tokens': 128,
            'prompt_cache_miss_tokens': 134
        },
        'model_provider': 'deepseek',
        'model_name': 'deepseek-flash',
        'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
        'id': 'efc445db-6751-4f2f-bbd1-ed150839dfa0',
        'finish_reason': 'tool_calls',
        'logprobs': None
    },
    id='lc_run--01a0c491-6559-7ce1-bd40-106475ad1f61-0',
    tool_calls=[
        {
            'name': 'get_weather',
            'args': {'city': '大连'},
            'id': 'call_00_gCt4xK92eISBb7Bdql5R1978',
            'type': 'tool_call'
        }
    ],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 262,
        'output_tokens': 38,
        'total_tokens': 300,
        'input_token_details': {'cache_read': 128},
        'output_token_details': {}
    }
)

## 2、工具描述的各部分详解

## 2.1 了解convert_to_openai_tool

执行`model.bind_tools([get_weather])`，底层最终会调用`convert_to_openai_tool`生成工具描述。所以我们可以直接调用后者查看解析后的工具描述。

In [2]:
from langchain_core.utils.function_calling import convert_to_openai_tool

def get_weather(city : str):
    return f"{city}天气晴朗~~"


rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

## 2.2 description说明

In [6]:
from langchain_core.utils.function_calling import convert_to_openai_tool

def get_weather(city : str):
    """
    查询城市的天气
    """
    return f"{city}天气晴朗~~"


rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '查询城市的天气',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

## 2.3 参数说明

In [10]:
from langchain_core.utils.function_calling import convert_to_openai_tool

def get_weather(city : str):
    """
    查询城市的天气

    Args:
        city : 具体的城市

    Returns:
        返回城市的天气
    """
    return f"{city}天气晴朗~~"


rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '查询城市的天气',
        'parameters': {
            'properties': {'city': {'description': '具体的城市', 'type': 'string'}},
            'required': ['city'],
            'type': 'object'
        }
    }
}

## 2.4 参数类型说明

举例1：正确的

In [12]:
from langchain_core.utils.function_calling import convert_to_openai_tool

def get_weather(city):
    """
    查询城市的天气
    """
    return f"{city}天气晴朗~~"


rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '查询城市的天气',
        'parameters': {'properties': {'city': {}}, 'required': ['city'], 'type': 'object'}
    }
}

举例2：如下的代码运行会报错

要求：如果在docstring中声明了参数的描述，则必须在函数声明处指明参数的类型

In [ ]:
from langchain_core.utils.function_calling import convert_to_openai_tool

def get_weather(city):
    """
    查询城市的天气

    Args:
        city : 具体的城市
    """
    return f"{city}天气晴朗~~"


rprint(convert_to_openai_tool(get_weather))

## 2.5 参数默认值说明

一旦参数设置的默认值，则打印的结果中的required字段中就不再包含此参数。

举例1：

In [14]:
from langchain_core.utils.function_calling import convert_to_openai_tool

def get_weather(city : str = "beijing"):
    """
    查询城市的天气

    Args:
        city : 具体的城市
    """
    return f"{city}天气晴朗~~"


rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '查询城市的天气',
        'parameters': {
            'properties': {'city': {'default': 'beijing', 'description': '具体的城市', 'type': 'string'}},
            'type': 'object'
        }
    }
}

举例2：

In [15]:
from langchain_core.utils.function_calling import convert_to_openai_tool

def get_weather(dt:str ,city : str = "beijing"):
    """
    查询城市的天气

    Args:
        city : 具体的城市
        dt : 时间
    """
    return f"{city}天气晴朗~~"


rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '查询城市的天气',
        'parameters': {
            'properties': {
                'dt': {'description': '时间', 'type': 'string'},
                'city': {'default': 'beijing', 'description': '具体的城市', 'type': 'string'}
            },
            'required': ['dt'],
            'type': 'object'
        }
    }
}